# Homework 5.2: RLVR on a real language model (GRPO on GSM8K)

In HW5.1 you wrote GRPO by hand for a 100k-parameter model.  In this notebook you will apply the
same algorithm to a real instruction-tuned LLM, **Qwen2.5-0.5B-Instruct**, on grade-school math
word problems (**GSM8K**), using Hugging Face's `trl` library.  Instead of writing the loop, you
will write the part that matters: the **reward functions**.

> **This notebook needs a GPU.**  A free Google Colab T4 is enough.  Training for 200 steps
> takes roughly an hour; budget accordingly and do not close the tab.  Nothing here will finish
> on a CPU.

**Learning Objectives:**
1. Prepare an RLVR dataset: prompts plus a *gold answer the verifier can check*, but no solutions to imitate.
2. Write a **correctness reward** and a **format reward**, and understand why practitioners combine them.
3. Read a GRPO training log: reward curves, completion length, KL.
4. Evaluate a model before and after RL with the *same* verifier, and relate what you see to the HW5.1 experiments.

**Your Task:**
* Implement the **TODO** in `correctness_reward()`.
* Implement the **TODO** in `format_reward()`.
* Run training, then run the evaluation cells, and answer the questions at the end.

**Credits.**  This notebook adapts Will Brown's GRPO demo
(W. Brown, *Granular Format Rewards for Eliciting Mathematical Reasoning Capabilities in Small
Language Models*, GitHub Gist, January 2025), which introduced the `<reasoning>`/`<answer>`
prompt, the answer-extraction helpers, and the split into a correctness reward and format
rewards used here.  The same recipe underlies the Hugging Face LLM course, chapter 12
("Practical Exercise: GRPO with Unsloth") and Unsloth's GRPO notebooks.  The trainer
configuration follows the `trl` GRPOTrainer documentation.  GSM8K is from Cobbe et al. (2021);
Qwen2.5 is from the Qwen team at Alibaba (2024).

In [ ]:
# Install dependencies (Colab).  Restart the runtime if pip asks you to.
!pip install -q "trl>=0.25" peft datasets accelerate

In [ ]:
import re
import torch
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer

assert torch.cuda.is_available(), "This notebook needs a GPU runtime (Runtime -> Change runtime type -> T4 GPU)."
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32
print("GPU:", torch.cuda.get_device_name(0), " dtype:", DTYPE)

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

## The dataset

GSM8K questions come with worked solutions that end in `#### <number>`.  For RLVR we keep only
the question (as a chat prompt) and the final number (as the gold answer for the verifier).  The
system prompt asks for a fixed output format so that the answer is easy to extract:

```
<reasoning>
...
</reasoning>
<answer>
...
</answer>
```

In [ ]:
SYSTEM_PROMPT = '''Respond in the following format:
<reasoning>
...
</reasoning>
<answer>
...
</answer>'''

def extract_xml_answer(text):
    '''Return the text inside the last <answer>...</answer> block, stripped.'''
    answer = text.split("<answer>")[-1]
    answer = answer.split("</answer>")[0]
    return answer.strip()

def extract_hash_answer(text):
    '''GSM8K solutions end with "#### 42".'''
    if "####" not in text:
        return None
    return text.split("####")[1].strip().replace(",", "").replace("$", "")

def build_prompt(question):
    return [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question}]

raw = load_dataset("openai/gsm8k", "main")
train_ds = raw["train"].map(lambda x: {"prompt": build_prompt(x["question"]), "answer": extract_hash_answer(x["answer"])})
test_ds = raw["test"].select(range(100)).map(lambda x: {"prompt": build_prompt(x["question"]), "answer": extract_hash_answer(x["answer"])})
print(train_ds[0]["prompt"][1]["content"])
print("gold answer:", train_ds[0]["answer"])

## Evaluating the base model

Before training, measure the model with the same verifier we will train against.  We generate
greedily for 100 test questions and record (a) whether the extracted answer is exactly right and
(b) whether the output followed the requested format.  Keep these numbers; you will compare
against them after RL.

In [ ]:
@torch.no_grad()
def generate_answers(model, prompts, max_new_tokens=256, batch_size=8):
    model.eval()
    outputs = []
    tokenizer.padding_side = "left"
    for i in range(0, len(prompts), batch_size):
        chats = [tokenizer.apply_chat_template(p, tokenize=False, add_generation_prompt=True) for p in prompts[i:i + batch_size]]
        enc = tokenizer(chats, return_tensors="pt", padding=True).to(model.device)
        gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        outputs += tokenizer.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return outputs

FORMAT_PATTERN = r"<reasoning>.*?</reasoning>\s*<answer>.*?</answer>"

def score(outputs, golds):
    correct = sum(extract_xml_answer(o) == g for o, g in zip(outputs, golds)) / len(golds)
    formatted = sum(bool(re.search(FORMAT_PATTERN, o, flags=re.DOTALL)) for o in outputs) / len(outputs)
    length = sum(len(tokenizer(o)["input_ids"]) for o in outputs) / len(outputs)
    return dict(accuracy=correct, format_rate=formatted, mean_tokens=length)

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=DTYPE).cuda()
base_outputs = generate_answers(base_model, test_ds["prompt"])
base_scores = score(base_outputs, test_ds["answer"])
print("BASE MODEL:", base_scores)
print("\nExample output:\n", base_outputs[0])
del base_model; torch.cuda.empty_cache()

## The reward functions

`trl` calls each reward function with the batch of sampled completions and every column of the
dataset as keyword arguments, so `answer` (the gold number) arrives as a list.  In the
conversational format each completion is a list with one message, so the text is
`completion[0]["content"]`.  Each function returns one float per completion; `trl` adds the
rewards from all functions together.

**TODO:** Implement `correctness_reward()` and `format_reward()`.

*Hints:*
- Correctness: `2.0` if `extract_xml_answer(text) == gold`, else `0.0`.
- Format: `0.5` if the text matches `FORMAT_PATTERN` (use `re.search` with `re.DOTALL`), else `0.0`.
- Why both?  In HW5.1 the model already used the required format some of the time, so a
  correctness reward alone had something to select.  A model that never produces `<answer>` tags
  earns zero correctness reward on every sample, every group has zero advantage, and nothing is
  learned.  The format reward gives a gradient before the first correct answer appears.

In [ ]:
def correctness_reward(completions, answer, **kwargs):
    '''2.0 for each completion whose extracted answer equals the gold answer, else 0.0.'''
    texts = [completion[0]["content"] for completion in completions]
    # ----- TODO: Student Implementation Starts Here -----
    # `texts` is the list of completion strings and `answer` is the list of gold answers (strings), same order.
    # Return a list of floats: 2.0 where extract_xml_answer(text) == gold, else 0.0.
    raise NotImplementedError("Implement correctness_reward")
    # ----- TODO: Student Implementation Ends Here -----

def format_reward(completions, **kwargs):
    '''0.5 for each completion that contains <reasoning>...</reasoning> followed by <answer>...</answer>.'''
    texts = [completion[0]["content"] for completion in completions]
    # ----- TODO: Student Implementation Starts Here -----
    # Return a list of floats: 0.5 where re.search(FORMAT_PATTERN, text, flags=re.DOTALL) finds a match, else 0.0.
    # (re.DOTALL lets "." match newlines, so the reasoning block may span several lines.)
    raise NotImplementedError("Implement format_reward")
    # ----- TODO: Student Implementation Ends Here -----

# Sanity checks
good = [[{"role": "assistant", "content": "<reasoning>\n2+2=4\n</reasoning>\n<answer>\n4\n</answer>"}]]
bad = [[{"role": "assistant", "content": "The answer is 4."}]]
assert correctness_reward(good, answer=["4"]) == [2.0] and correctness_reward(good, answer=["5"]) == [0.0]
assert format_reward(good) == [0.5] and format_reward(bad) == [0.0]
print("reward functions OK")

## Training with GRPO

The configuration below samples 8 completions per prompt, takes one prompt per step, and trains
a LoRA adapter rather than the full model so that everything fits on a T4.  `beta` is the KL
coefficient from HW5.1; `trl` uses the same k3 estimator.  Two hundred steps is enough to see the
reward move; if you have the time, more is better.

If you see `CUDA out of memory`, halve `max_completion_length` or `num_generations`.

In [ ]:
training_args = GRPOConfig(
    output_dir="outputs/qwen0.5b-grpo-gsm8k",
    learning_rate=2e-5,             # LoRA tolerates a larger learning rate than full fine-tuning
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    max_grad_norm=0.1,
    per_device_train_batch_size=8,  # must be a multiple of num_generations
    gradient_accumulation_steps=1,
    num_generations=8,              # G in HW5.1
    max_prompt_length=256,
    max_completion_length=256,
    beta=0.04,                      # KL coefficient
    max_steps=200,
    logging_steps=5,
    log_completions=True,
    save_strategy="no",
    report_to="none",
    bf16=(DTYPE == torch.bfloat16),
)

peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

trainer = GRPOTrainer(
    model=MODEL_NAME,
    processing_class=tokenizer,
    reward_funcs=[format_reward, correctness_reward],
    args=training_args,
    train_dataset=train_ds,
    peft_config=peft_config,
)
trainer.train()

## Reading the training log

`trl` records the mean of each reward function, the total reward, the completion length, and the
KL at every logging step.  Plot them.

In [ ]:
logs = [row for row in trainer.state.log_history if "reward" in row]
steps = [row["step"] for row in logs]
keys = sorted(k for k in logs[0] if k.startswith("reward") or k.startswith("completion") or k == "kl")
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for k in keys:
    if k.startswith("reward"):
        axes[0].plot(steps, [row.get(k) for row in logs], label=k)
axes[0].set_title("rewards"); axes[0].legend(fontsize=7)
for k in keys:
    if k.startswith("completion") and "length" in k and "mean" in k:
        axes[1].plot(steps, [row.get(k) for row in logs], label=k)
axes[1].set_title("completion length"); axes[1].legend(fontsize=7)
if "kl" in keys:
    axes[2].plot(steps, [row.get("kl") for row in logs]); axes[2].set_title("KL to reference")
for ax in axes:
    ax.set_xlabel("step")
plt.tight_layout(); plt.show()

## Evaluating the trained model

Run the same evaluation as before on the same 100 questions.

In [ ]:
rl_model = trainer.model
rl_outputs = generate_answers(rl_model, test_ds["prompt"])
rl_scores = score(rl_outputs, test_ds["answer"])
print("BASE MODEL: ", base_scores)
print("AFTER GRPO: ", rl_scores)
print("\nExample output after RL:\n", rl_outputs[0])

### Questions (answer in this cell)

1. Report the before/after table (accuracy, format rate, mean tokens).  Which moved first during
training, the format reward or the correctness reward?  Why is that the expected order?

    *Your answer:*

2. In HW5.1 pass@16 did not change after RL.  Design (do not run) an experiment that would test
whether the same is true here, and say what result would convince you that RL taught the model
something *new* rather than sharpening what it already knew.

    *Your answer:*

3. Look at five completions from late in training (the trainer prints them when
`log_completions=True`).  Do you see any sign of reward hacking: answers that satisfy the
verifier without solving the problem, format tricks, or repeated text?  If you had to make
`correctness_reward` more robust, what would you change?

    *Your answer:*

4. Estimate the cost of this run: how many completions were sampled in total, and how many
tokens were generated?  Compare with the number of training tokens a supervised fine-tuning run
on the same 200 questions would use.  What does this say about where the compute goes in RLVR?

    *Your answer:*